In [1]:
import re
import pandas as pd
import json
import csv
import os
import traceback
import numpy as np 

from collections import Counter
from typing import List

pd.options.mode.chained_assignment = None

In [2]:
ad_master_file = 'ad_master.csv'
db = pd.read_csv(ad_master_file)

advertiser_scam_count = 'advertiser_scam_count.csv'
advertiser_db = pd.read_csv(advertiser_scam_count)

In [4]:
def match(url):
    pattern = r"(?<=watch\?v=).{11}"
    match = re.search(pattern, url)
    return match[0]

def boolify(str) -> bool:
    return str.lower() == 'true'

def get_label(ad_id: str, db: pd.DataFrame) -> tuple:
    
    match = db[db["preroll_ad_id"]==ad_id]
    
    if not len(match):
        return "", "", ""
    
    note = match[match['Notes'].notna()]
    if len(note):
        note = note.iloc[0]["Notes"]
    else:
        note = ""
    
    match = match.iloc[0]
    
    tag = "" if not match['tag'] else match['tag']
    is_scam = int(match["is_scam"])
    is_scam_2 = match["is_scam_2"]
    annotator_1 = match["annotator_1"]
    annotator_2 = match["annotator_2"]
    
    return is_scam, tag, note, is_scam_2, annotator_1, annotator_2

def get_label_advertiser(advertiser: str, advertiser_db: pd.DataFrame):
    
    if not advertiser:
        return ""
    
    advertiser = advertiser.lower()
    match = advertiser_db[advertiser_db['preroll_ad_advertiser'].astype(str).str.lower()==advertiser]
    
    if not len(match):
        return ""
    
    if len(match) > 1:
        match.sort_values(by=['total'], ascending=False)
        
    match = match.iloc[0]
    
    if match['total'] < 20:
        return ""
    
    if match['scam_video_count'] != 0:
        return ""
    return 0
    

def compute_labeled_run(df: pd.DataFrame) -> tuple:
    # used for completed runs, df is all rows, otherwise might throw error
    # filter out pr1 and pr2, match with database, check for completion, then calculate scam/kid rate
    # output pr1_scam, pr2_scam, pr_scam, pr_kid_rate
    
    df_pr1 = df[df['preroll_ad_id'].notna()]
    df_pr2 = df[df['preroll_ad2_id'].notna()]
    
    pr1_count = len(df_pr1)
    pr2_count = len(df_pr2)
    pr_count = pr1_count + pr2_count
    
    df_pr1['is_scam'] = df_pr1["preroll_ad_id"].apply(lambda x: get_label(x, db)[0])
    df_pr2['is_scam'] = df_pr2["preroll_ad2_id"].apply(lambda x: get_label(x, db)[0])
    
    #df_pr1['is_kid'] = df_pr1["preroll_ad_id"].apply(lambda x: get_label(x, db)[1])
    #df_pr2['is_kid'] = df_pr2["preroll_ad2_id"].apply(lambda x: get_label(x, db)[1])
    
    df_pr1_labeled = df_pr1[df_pr1['is_scam'] != '']
    df_pr2_labeled = df_pr2[df_pr2['is_scam'] != '']
       
    pr1_has_label_count = len(df_pr1_labeled)
    #pr1_has_kids = df_pr1['is_kid'].count()
    
    pr2_has_label_count = len(df_pr2_labeled)
    #pr2_has_kids = df_pr2['is_kid'].count()
    
    print('unlabeled pr1:')
    df_pr1_unlabeled = df_pr1[df_pr1['is_scam'] == '']
    display(df_pr1_unlabeled['preroll_ad_id'])
    
    print('unlabeled pr2:')
    df_pr2_unlabeled = df_pr2[df_pr2['is_scam'] == '']
    display(df_pr2_unlabeled['preroll_ad2_id'])
    
    pr_has_label_count = pr1_has_label_count + pr2_has_label_count
    
    print(f'Missing scam rows for pr1: {pr1_count-pr1_has_label_count}')
    #print(f'Missing kids rows for pr1: {pr1_count-pr1_has_kids}')
    print(f'Missing scam rows for pr2: {pr2_count-pr2_has_label_count}')
    #print(f'Missing kids rows for pr2: {pr2_count-pr2_has_kids}')
    
    pr1_scam_count = sum(df_pr1_labeled['is_scam'])
    pr1_scam_rate = pr1_scam_count/pr1_has_label_count
    
    pr2_scam_count = sum(df_pr2_labeled['is_scam']) 
    pr2_scam_rate = pr2_scam_count/pr2_has_label_count
    
    pr_scam_count = pr1_scam_count + pr2_scam_count
    pr_scam_rate = pr_scam_count/pr_has_label_count
    
    #pr_kid_count = sum(df_pr1['is_kid']) + sum(df_pr2['is_kid'])
    #pr_kid_rate = pr_kid_count / pr_count
    
    print(f"Preroll ads: {pr_count}")
    print(f"Preroll 1 ads: {pr1_count}")
    print(f"Preroll 2 ads: {pr2_count}")
    print(f"Preroll 1 scam rate: {pr1_scam_count}/{pr1_has_label_count} = {pr1_scam_rate:.4f}")
    print(f"Preroll 2 scam rate: {pr2_scam_count}/{pr2_has_label_count} = {pr2_scam_rate:.4f}")
    print(f"Preroll scam rate: {pr_scam_count}/{pr_has_label_count} = {pr_scam_rate:.4f}")
    #print(f"Preroll kid rate: {pr_kid_count}/{pr_count} = {pr_kid_rate:.4f}")
    
    return pr1_scam_rate, pr2_scam_rate, pr_scam_rate

def parse_file(folder: str, id: str, no_ad_mode: bool, verbose: bool) -> pd.DataFrame:
    ads_rows = []
    all_rows = []
    run_attributes = []

    unavailable_count = 0
    available_count = 0

    ad_attributes = ["preroll_ad_id", "preroll_ad2_id", "side_ad_site", "side_ad_img", "promoted_video_title"]

    with open(f"{folder}/output_{id}.json", 'r', encoding="utf-8") as f:
        for _ in range(4):
            line = f.readline()
            arr = line.split(',')
            run_attributes.extend(arr)

        for line in f:
            line = line.replace("This link opens in new tab", "")
            json_data = json.loads(line)
            
            if no_ad_mode:
                if not "video_title" in json_data:
                    unavailable_count += 1
                elif json_data["is_for_kids"] == True:
                    unavailable_count += 1
                else:
                    available_count += 1
                    all_rows.append(json_data)
            else:
                if not "video_title" in json_data or not (json_data["video_title"] or json_data["preroll_ad_id"]):
                    unavailable_count += 1
                elif json_data["is_for_kids"] == True:
                    unavailable_count += 1
                else:
                    available_count += 1
                    all_rows.append(json_data)

    df = pd.DataFrame(all_rows)
    
    if verbose:
        print(f"{id}")
        gender = run_attributes[2].strip()
        video_list = run_attributes[3].split('/')[1]
        until_available = boolify(run_attributes[4].split(':')[1].strip())
        random = boolify(run_attributes[5].split(':')[1].strip())
        ad_pers = boolify(run_attributes[6].split(':')[1].strip())
        act_on = boolify(run_attributes[7].split(':')[1].strip())
        # sleep = int(run_attributes[10].split(':')[1].strip().replace('s', ''))
    
        print(gender)
        print(video_list)
        print(f"until_available: {until_available}")
        print(f"random: {random}")
        print(f"ad_pers: {ad_pers}")
        print(f"act_on: {act_on}")
        # print(f"sleep (in seconds): {sleep}")
        print()

        if not no_ad_mode:
            try:
                for_kids_count = sum(df['is_for_kids'])
                age_restricted_count = sum(df['is_age_restricted'])
            except KeyError:
                pass
            
            preroll1_count = df['preroll_ad_id'].count()
            preroll2_count = df['preroll_ad2_id'].count()
            preroll_count = preroll1_count + preroll2_count
            side_count = df['side_ad_img'].count()
            promoted_count = df['promoted_video_title'].count()
            available_count = df['video_title'].count()
            ad_count = len(ads_rows)

            print(f"Unavailable videos: {unavailable_count}")
            print(f"Available videos: {available_count}")
            print(f"Preroll ads: {preroll_count}")
            print(f"Preroll 1 ads: {preroll1_count}")
            print(f"Preroll 2 ads: {preroll2_count}")
            print(f"Side ads: {side_count}")
            print(f"Promoted videos: {promoted_count}")
            print(f"Number of ads: {preroll_count+side_count+promoted_count}")
            print(f"Number of videos with one or more ads: {ad_count}")

            try:
                print(f"Number of videos that are for kids: {for_kids_count}")
                print(f"Number of videos that are age restricted: {age_restricted_count}")
            except KeyError:
                pass

        else:
            try:
                for_kids_count = sum(df['is_for_kids'])
                age_restricted_count = sum(df['is_age_restricted'])
            except KeyError:
                pass

            print(f"Unavailable videos: {unavailable_count}")
            print(f"Available videos: {available_count}")

            try:
                print(f"Number of videos that are for kids: {for_kids_count}")
                print(f"Number of videos that are age restricted: {age_restricted_count}")
            except:
                pass
            
    return df

def parse_from_df(df: pd.DataFrame) -> pd.DataFrame:
    
    # return the original df
    
    try:
        for_kids_count = sum(df['is_for_kids'])
        age_restricted_count = sum(df['is_age_restricted'])
    except KeyError:
        pass

    preroll1_count = df['preroll_ad_id'].count()
    preroll2_count = df['preroll_ad2_id'].count()
    preroll_count = preroll1_count + preroll2_count
    side_count = df['side_ad_img'].count()
    promoted_count = df['promoted_video_title'].count()
    available_count = df['video_title'].count()

    print(f"Preroll ads: {preroll_count}")
    print(f"Preroll 1 ads: {preroll1_count}")
    print(f"Preroll 2 ads: {preroll2_count}")
    print(f"Side ads: {side_count}")
    print(f"Promoted videos: {promoted_count}")
    print(f"Number of ads: {preroll_count+side_count+promoted_count}")

    try:
        print(f"Number of videos that are for kids: {for_kids_count}")
        print(f"Number of videos that are age restricted: {age_restricted_count}")
    except KeyError:
        pass
    
    return df
    

def melt_and_match_preroll(df: pd.DataFrame) -> pd.DataFrame:
    
    # melt preroll_ad_id i.e. create extra rows
    value_vars = ['preroll_ad_id', 'preroll_ad2_id']
    id_vars = [i for i in df.columns if i not in value_vars]
    df_flat = df.melt(id_vars=id_vars, value_vars=value_vars, 
                      var_name='original_id', value_name='pr_ad_id')

    # filter out rows without any preroll ad
    df_flat = df_flat[df_flat['pr_ad_id'].notnull()]

    # map ad order
    df_flat['ad_order'] = df_flat['original_id'].map({'preroll_ad_id': 1, 'preroll_ad2_id': 2})
    df_flat = df_flat.drop(columns=['original_id'])
    df_flat.rename(columns={'pr_ad_id':'preroll_ad_id'}, inplace=True)

    # nullifying irrelevant columns in order to merge
    cols_null1 = ['preroll_ad_video_url', 'preroll_ad_duration', 'preroll_ad_site', 'preroll_ad_link',
                 'preroll_ad_card', 'preroll_ad_reasons', 'preroll_ad_info','preroll_ad_topic']
    df_flat.loc[df_flat['ad_order'] == 2, cols_null1] = np.nan

    cols_null2 = ['preroll_ad2_video_url', 'preroll_ad2_duration', 'preroll_ad2_site', 'preroll_ad2_link',
                 'preroll_ad2_card', 'preroll_ad2_reasons', 'preroll_ad2_info', 'preroll_ad2_topic']
    df_flat.loc[df_flat['ad_order'] == 1, cols_null2] = np.nan

    for col_null1, col_null2 in zip(cols_null1, cols_null2):
        df_flat[col_null1] = df_flat[col_null1].combine_first(df_flat[col_null2])

    df_flat.drop(cols_null2, axis=1, inplace=True)

    # match label
    df_flat['preroll_is_scam'] = df_flat['preroll_ad_id'].apply(lambda x: get_label(x, db)[0])
    
    # merge with rows with no ads
    df_noad = df[df['preroll_ad_id'].isnull()]
    df_noad.drop([col for col in df_noad.columns if col not in df_flat.columns], axis=1, inplace=True)

    final = pd.concat([df_flat, df_noad], ignore_index=True)
    return final

def parse_id(filename: str) -> str:
    pattern = r"(\d{4}_\d{4}_.+(?=\.))|(\d{8}_\d{4}_.+(?=\.))"
    id = re.search(pattern, filename)[0]
    return id

def conv_video(video_set: str) -> str:
    # match video set from www24 to general text
    v = {
        'science_videos_subset_20230815_0_30_41.csv': "science",
        'us_entertainment_unique_Aug152023.csv': "entertainment",
        'faddoul_available_Aug152023.csv': "conspiracy",
        'news_subset_Sep212023.csv': "news"
    }
    return v[video_set]

def gen_preroll_wip(df: pd.DataFrame, save: bool, **kwargs) -> pd.DataFrame:
    # flatten preroll ad 1 and 2
    df_preroll1 = df[df['preroll_ad_id'].notna()]
    df_preroll2 = df[df['preroll_ad2_id'].notna()]

    pr1_cols = ['preroll_ad_id', 'preroll_ad_video_url', 'preroll_ad_site',
                'preroll_ad_card', 'preroll_ad_info', 'preroll_ad_topic']
    
    pr2_cols = ['preroll_ad2_id', 'preroll_ad2_video_url', 'preroll_ad2_site',
                'preroll_ad2_card', 'preroll_ad2_info', 'preroll_ad2_topic']

    # drop unnecessary columns for clean output
    df_preroll1.drop([c for c in df_preroll1.columns if c not in pr1_cols], axis=1, inplace=True)
    df_preroll2.drop([c for c in df_preroll2.columns if c not in pr2_cols], axis=1, inplace=True)

    df_preroll2.columns = pr1_cols

    df_preroll1['ad_order'] = [1]*len(df_preroll1)
    df_preroll2['ad_order'] = [2]*len(df_preroll2)

    df_preroll = pd.concat([df_preroll1, df_preroll2]).reset_index(drop=True)

    df_preroll['preroll_ad_advertiser'] = df_preroll['preroll_ad_info'].apply(lambda x:x[0] if x else "")
    df_preroll['preroll_ad_location'] = df_preroll['preroll_ad_info'].apply(lambda x:x[1] if x else "")
    df_preroll.drop('preroll_ad_info', axis=1, inplace=True)

    # preroll - combine duplicates and count occurences

    df = df_preroll
    df['occurences'] = df.groupby('preroll_ad_id')['preroll_ad_id'].transform('count')

    df['pr1_count'] = df.groupby(['preroll_ad_id'])['ad_order'].transform(lambda x: (x==1).sum())
    df['pr2_count'] = df.groupby(['preroll_ad_id'])['ad_order'].transform(lambda x: (x==2).sum())

    df = df.sort_values(by=['occurences'], ascending=False)
    df = df.drop_duplicates(subset=['preroll_ad_id']).reset_index(drop=True)
    df_preroll.drop('ad_order', axis=1, inplace=True)
    
    # match ad id to ad database
    result = df["preroll_ad_id"].apply(lambda x: get_label(x, db))
    # return is_scam, tag, note, is_scam_2, annotator_1, annotator_2
    
    dft = pd.DataFrame(result.tolist())
    df["is_scam"] = dft[0]
    df["annotator_1"] = dft[4]
    df["is_scam_2"] = dft[3]
    df["annotator_2"] = dft[5]
    # df["tag"] = dft[1]
    df["is_scam_new"] = ""
    df["Notes"] = dft[2]
    
    
    # match advertiser and auto label
    
    # Identify rows where is_scam is empty
    mask = df['is_scam'] == ""
    df.loc[mask, 'is_scam'] = df.loc[mask, 'preroll_ad_advertiser'].apply(lambda x: get_label_advertiser(x, advertiser_db))
    df.loc[mask & df['is_scam'].isin([0, 1]), 'annotator_1'] = 'automatch_advertiser'
    
    # Sort ad site alphabetically
    df = df.sort_values('preroll_ad_site')
    
    if save:
        ad_dir = kwargs.get('ad_dir', 'ad_dir')
        id = kwargs.get('id', 'id') 
        df.to_csv(f'{ad_dir}/preroll_wip_{id}.csv', index=False)
    return df
    
def gen_side_wip(df: pd.DataFrame, ad_dir: str, id: str, save: bool) -> pd.DataFrame:
    
    # output side ads

    df_side = df[df['side_ad_img'].notna()]

    dedupe_col = 'side_ad_img'

    # drop unnecessary columns for clean output
    side_cols = ['side_ad_site', 'side_ad_text', 'side_ad_img','side_ad_reasons',
                 'side_ad_info', 'side_ad_topic', 'side_ad_link']
    df_side.drop([c for c in df_side.columns if c not in side_cols], axis=1, inplace=True)
    df_side['side_occurences'] = df_side.groupby(dedupe_col)[dedupe_col].transform('count')

    df_side = df_side.sort_values(by=['side_occurences'], ascending=False)
    df_side = df_side.drop_duplicates(subset=[dedupe_col]).reset_index(drop=True)

    # split ad info since it's a list
    df_side['side_ad_advertiser'] = df_side['side_ad_info'].apply(lambda x:x[0] if x else "")
    df_side['side_ad_location'] = df_side['side_ad_info'].apply(lambda x:x[1] if x else "")
    df_side.drop('side_ad_info', axis=1, inplace=True)
    
    if save:
        df_side.to_csv(f'{ad_dir}/side_wip_{id}.csv', index=False)
    return df_side

def to_country(id_str: str) -> str:
    if 'aus' in id_str:
        return 'aus'
    if 'ire' in id_str or 'irl' in id_str:
        return 'irl'
    if 'cnd' in id_str or 'can' in id_str:
        return 'can'
    if 'usa' in id_str or 'west' in id_str:
        return 'usa'
    if 'gbr' in id_str:
        return 'gbr'

NameError: name 'pd' is not defined